# Reranker 반환 시간 비교 (Colab GPU)

`bge-reranker-v2-m3`(cross-encoder) vs `bge-reranker-v2-gemma`(LLM-based)의 **rerank 반환 시간**만 비교한다.

**실행 전**: 런타임 → 런타임 유형 변경 → **T4 GPU**.

> - m3: `sentence-transformers`의 `CrossEncoder`로 로드 (최신 transformers 호환)
> - gemma: `FlagLLMReranker`. 단 최신 transformers는 `prepare_for_model`을 제거했으므로 **transformers==4.48.3** 으로 핀한다.
> - ⚠️ 설치 셀이 transformers를 바꾸므로, 설치 후 **런타임 → 세션 다시 시작** 한 뒤 모두 실행할 것.

## 1) GPU 확인

In [1]:
import torch
print('CUDA 사용 가능:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM(GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
else:
    print('⚠️ GPU 미할당 — 런타임 유형을 T4 GPU로 바꾸고 다시 실행하세요.')

CUDA 사용 가능: True
GPU: Tesla T4
VRAM(GB): 14.6


## 2) 설치
- `sentence-transformers`: m3(cross-encoder)용
- `FlagEmbedding`: gemma(LLM reranker)용
- `transformers==4.48.3`: gemma의 prepare_for_model 오류 회피 (4.49에서 제거됨)

**이 셀 실행 후 → 런타임 → 세션 다시 시작 → 1번부터 모두 실행.**

In [2]:
!pip install -q sentence-transformers FlagEmbedding transformers==4.48.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 117.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 114.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 50.9 MB/s eta 0:00:00


## 3) 설정 + 샘플 데이터

샘플 passage **5개**를 각각 `REPEAT`회 복제해 후보 수를 만든다 (5 × 10 = 50).
실제 비교를 원하면 `SAMPLE_PASSAGES`를 본인 도메인 실제 청크로 교체하세요. 토큰 길이가 latency를 좌우합니다.

In [3]:
# ── 측정 설정 ──
REPEAT = 10         # 각 sample passage를 몇 번 복제할지 (5개 × 10 = 50)
ITERATIONS = 20     # 측정 반복
WARMUP = 3          # 예열(측정 제외)
BATCH_SIZE = 16     # 내부 배치
RUN_GEMMA = True    # gemma도 측정할지 (False면 m3만)

SAMPLE_QUERY = 'FastAPI에서 비동기 미들웨어는 어떻게 동작하나?'

# 정확히 5개
SAMPLE_PASSAGES = [
    'FastAPI의 미들웨어는 모든 요청과 응답을 가로채는 계층이다. async def로 정의하면 비동기 처리되며, '
    'call_next(request)를 await 하여 다음 핸들러로 제어를 넘긴다. 요청 전후에 로깅·인증·헤더 주입 같은 공통 처리를 둘 수 있다.',
    'ASGI는 비동기 서버 게이트웨이 인터페이스로, FastAPI가 동작하는 기반이다. '
    'WSGI와 달리 async/await를 네이티브로 지원하여 동시 연결을 효율적으로 처리한다.',
    '데이터베이스 연결 풀은 동시 요청이 많을 때 커넥션을 재사용해 오버헤드를 줄인다. '
    '비동기 드라이버(asyncpg 등)를 쓰면 이벤트 루프를 막지 않고 쿼리를 수행한다.',
    'Pydantic 모델은 요청 본문을 검증하고 타입을 강제한다. FastAPI는 이를 통해 자동 문서화(OpenAPI)와 직렬화를 제공한다. 미들웨어와는 직접 관련이 없다.',
    '백그라운드 작업은 BackgroundTasks로 응답 이후에 실행할 수 있다. 무거운 작업은 별도 워커(Celery 등)로 분리하는 것이 권장된다. 미들웨어 단계와는 별개다.',
]


def preprocess_query(q):
    """간단한 질의 전처리 — 공백 정리 정도만(neural 모델엔 과한 구두점 제거 비권장)."""
    return ' '.join(q.split())


def build_pairs(query, passages, repeat):
    """각 passage를 repeat회 복제해 질의-후보 쌍 생성 (5개 × 10 = 50쌍)."""
    return [[query, p] for p in passages for _ in range(repeat)]


TOP_K = len(SAMPLE_PASSAGES) * REPEAT  # = 50
print(f'설정 완료. passage {len(SAMPLE_PASSAGES)}개 × {REPEAT} = top-k {TOP_K}')

설정 완료. passage 5개 × 10 = top-k 50


## 4) 측정 함수

In [4]:
import time, statistics


def measure(score_fn, iterations, warmup):
    """warmup 후 iterations 회 점수 계산 시간을 ms로 측정. score_fn은 인자 없는 호출."""
    for _ in range(warmup):
        score_fn()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
    times = []
    for _ in range(iterations):
        t0 = time.perf_counter()
        score_fn()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000.0)
    return times


def summarize(times, top_k):
    """ms 리스트 → 평균/중앙값/p95/문서당."""
    ordered = sorted(times)
    p95 = ordered[max(0, int(len(ordered) * 0.95) - 1)]
    mean = statistics.mean(times)
    return {
        'total_mean_ms': round(mean, 1),
        'total_median_ms': round(statistics.median(times), 1),
        'total_p95_ms': round(p95, 1),
        'per_doc_mean_ms': round(mean / top_k, 2),
    }


query = preprocess_query(SAMPLE_QUERY)
pairs = build_pairs(query, SAMPLE_PASSAGES, REPEAT)
results = []
print('쌍 개수:', len(pairs))

쌍 개수: 50


## 5) m3 측정 (cross-encoder, sentence-transformers CrossEncoder)

In [5]:
from sentence_transformers import CrossEncoder

t0 = time.perf_counter()
m3 = CrossEncoder('BAAI/bge-reranker-v2-m3', max_length=512)
print(f'm3 로딩: {(time.perf_counter()-t0)*1000:,.0f} ms')

s = summarize(measure(lambda: m3.predict(pairs, batch_size=BATCH_SIZE), ITERATIONS, WARMUP), TOP_K)
s['name'] = 'bge-reranker-v2-m3 (cross-encoder)'
results.append(s)
print(f"rerank(top-{TOP_K})  평균 {s['total_mean_ms']:,} ms | 중앙값 {s['total_median_ms']:,} ms | p95 {s['total_p95_ms']:,} ms")
print(f"문서당 평균 {s['per_doc_mean_ms']} ms")

del m3
torch.cuda.empty_cache()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

m3 로딩: 28,369 ms
rerank(top-50)  평균 709.2 ms | 중앙값 709.4 ms | p95 731.0 ms
문서당 평균 14.18 ms


## 6) gemma 측정 (LLM-based, FlagLLMReranker)

In [6]:
if RUN_GEMMA:
    from FlagEmbedding import FlagLLMReranker
    t0 = time.perf_counter()
    gemma = FlagLLMReranker('BAAI/bge-reranker-v2-gemma', use_fp16=True)
    print(f'gemma 로딩: {(time.perf_counter()-t0)*1000:,.0f} ms')

    s = summarize(measure(lambda: gemma.compute_score(pairs, batch_size=BATCH_SIZE), ITERATIONS, WARMUP), TOP_K)
    s['name'] = 'bge-reranker-v2-gemma (LLM-based)'
    results.append(s)
    print(f"rerank(top-{TOP_K})  평균 {s['total_mean_ms']:,} ms | 중앙값 {s['total_median_ms']:,} ms | p95 {s['total_p95_ms']:,} ms")
    print(f"문서당 평균 {s['per_doc_mean_ms']} ms")

    del gemma
    torch.cuda.empty_cache()
else:
    print('RUN_GEMMA=False — gemma 측정 건너뜀')

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/555 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

gemma 로딩: 108,531 ms


pre tokenize: 100%|██████████| 4/4 [00:00<00:00, 242.75it/s]
You're using a GemmaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
100%|██████████| 4/4 [00:02<00:00,  1.47it/s]


rerank(top-50)  평균 2,520.4 ms | 중앙값 2,492.6 ms | p95 2,735.2 ms
문서당 평균 50.41 ms


## 7) 비교 결과

In [7]:
print(f"{'모델':<38}{'평균(ms)':>10}{'문서당(ms)':>12}")
print('-' * 60)
for r in results:
    print(f"{r['name']:<38}{r['total_mean_ms']:>10,}{r['per_doc_mean_ms']:>12}")
if len(results) == 2:
    ratio = results[1]['total_mean_ms'] / results[0]['total_mean_ms']
    print('-' * 60)
    print(f'gemma가 m3보다 약 {ratio:.1f}배 {"느림" if ratio >= 1 else "빠름"}')

모델                                        평균(ms)     문서당(ms)
------------------------------------------------------------
bge-reranker-v2-m3 (cross-encoder)         709.2       14.18
bge-reranker-v2-gemma (LLM-based)        2,520.4       50.41
------------------------------------------------------------
gemma가 m3보다 약 3.6배 느림
